# Laboratorio 3 — Teoría: NumPy, arreglos y tensores 🎨

IA7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026

---

### Cuerpo Docente

- Profesor: Pablo Badilla y Ignacio Nuñez
- Auxiliar: Sofía Chávez
- Ayudantes: Javiera Arévalo, Tamara Carrasco, Ignacio Reyes

### Objetivos de este notebook

- Entender qué es un **arreglo** y qué es un **tensor**, y cómo se leen sus ejes.
- Aplicar vectorización, indexado, slicing, indexado condicional, broadcasting y
  reducciones por eje.
- Distinguir una **vista** de una **copia**, y por qué eso genera bugs difíciles.
- Reconocer que una imagen es un tensor más, para llegar preparados al mini
  proyecto del enunciado.

Este notebook es **solo teoría**: no se entrega y no tiene puntaje. Ejecútenlo
de principio a fin, modifiquen los valores y rompan las celdas a propósito —
es la forma de que las técnicas queden aprendidas antes de aplicarlas.

El mini proyecto está en `Lab3_Enunciado.ipynb`.

---

## El hilo conductor: una muestra meteorológica

Todo el notebook trabaja sobre una misma muestra de datos que registra
**3 sensores** (temperatura, humedad y viento), **cada hora**, durante **una
semana**.

Esos datos forman un bloque de números con tres ejes:

```text
(7 días, 24 horas, 3 sensores)
```

Guarden bien esa forma. Al final del notebook van a ver que una fotografía
a color tiene exactamente la misma estructura —`(alto, ancho, 3 canales)`— y
que todo lo que aprendan acá se aplica tal cual sobre una imagen.

### Mapa de los ejes

Una forma de tres ejes se entiende mejor con una muestra pequeña. Cada terna está ordenada como temperatura, humedad y viento.

```mermaid
block-beta
  columns 2

  block:dia0
    columns 4
    d0_title["Axis 0 = 0 (Día 0)"]:4
    h0["Hora"]:1 m0["M0"]:1 m1["M1"]:1 m2["M2"]:1
    h00["h0"]:1 v00["14"]:1 v01["82"]:1 v02["5"]:1
    h01["h1"]:1 v10["19"]:1 v11["68"]:1 v12["11"]:1
    h02["h2"]:1 v20["23"]:1 v21["55"]:1 v22["18"]:1
  end

  block:dia1
    columns 4
    d1_title["Axis 0 = 1 (Día 1)"]:4
    h1["Hora"]:1 n0["M0"]:1 n1["M1"]:1 n2["M2"]:1
    h10["h0"]:1 w00["13"]:1 w01["86"]:1 w02["3"]:1
    h11["h1"]:1 w10["18"]:1 w11["71"]:1 w12["9"]:1
    h12["h2"]:1 w20["24"]:1 w21["52"]:1 w22["21"]:1
  end
```

Por ejemplo, `muestra[1, 2, 0]` entra al bloque del día 1, busca la hora 2 y toma el sensor 0: la temperatura `24`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

---

## 1. Arreglos y tensores

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — arreglo y tensor</strong>
  Un <strong>arreglo</strong> es una zona de almacenamiento contiguo en memoria que contiene una serie de elementos del mismo tipo. En NumPy, el objeto que encapsula arreglos n-dimensionales se llama <code>ndarray</code>. Los arreglos de más de dos dimensiones reciben comúnmente el nombre de <strong>tensores</strong>.
</div>

La cantidad de ejes crece según cuánta información quieran guardar:

| Qué guardan | Ejes | Forma | Ejemplo |
|---|---|---|---|
| Una lectura suelta | 0 | `()` | la temperatura ahora |
| Un sensor durante un día con mediciones cada 6 horas | 1 | `(4,)` | 3 temperaturas |
| Los 3 sensores (temperatura, humedad y viento) durante un día | 2 | `(4, 3)` | una matriz |
| Los 3 sensores durante una semana | 3 | `(7, 4, 3)` | un **tensor** |

Empecemos con una muestra chica, escrita a mano, para poder contarla con la
vista: **2 días × 3 horas × 3 sensores**.

In [ ]:
import numpy as np

import numpy as np

# Ejemplo: Muestra meteorológica de Valparaíso
# Forma (4, 4, 3): 4 días, 4 lecturas diarias (00:00, 06:00, 12:00, 18:00), 3 métricas
# Métricas por lectura: [Temperatura °C, Humedad %, Viento km/h]

muestra = np.array(
    [
        # Día 0
        [
            [11, 92, 12],  # 00:00
            [10, 95, 8],   # 06:00 (Vaguada/camanchaca matutina)
            [16, 72, 18],  # 12:00
            [14, 83, 24]   # 18:00 (Viento costero del suroeste tarde)
        ],
        # Día 1
        [
            [12, 90, 10],  # 00:00
            [11, 93, 7],   # 06:00
            [17, 68, 22],  # 12:00
            [15, 78, 28]   # 18:00
        ],
        # Día 2
        [
            [10, 94, 9],   # 00:00
            [9, 96, 6],    # 06:00
            [15, 75, 16],  # 12:00
            [13, 85, 20]   # 18:00
        ],
        # Día 3
        [
            [11, 91, 14],  # 00:00
            [10, 94, 10],  # 06:00
            [18, 65, 26],  # 12:00
            [14, 80, 30]   # 18:00
        ]
    ]
)

print(muestra.shape)  # Confirmar dimensiones: (4, 4, 3)
print("Shape: ", muestra.shape)

muestra

Lean la forma de afuera hacia adentro: 2 bloques (días), cada uno con 3 filas
(horas), y cada fila con 3 números (sensores).

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — ubicar una lectura</strong>
  <code>muestra[1, 2, 0]</code> es la temperatura (sensor <code>0</code>) del día <code>1</code> a la hora <code>2</code>. Antes de ejecutar la celda, búsquenla con la vista en el arreglo de arriba.
</div>

In [ ]:
print("Día 1 | Hora 2 | Temperatura:", muestra[1, 2, 0])
print("Día 0 completo:\n", muestra[0])


---

## 2. Propiedades de un `ndarray`

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — axes</strong>
  En NumPy, las dimensiones se llaman <em>axes</em>. Un punto <code>[0, 0]</code> tiene un eje; una matriz <code>[[0, 0], [1, 2]]</code> tiene dos; y la muestra <code>(7, 24, 3)</code> tiene tres. El <strong>eje 0</strong> es el más externo.
</div>

| Atributo | Qué reporta | Sobre la muestra |
|---|---|---|
| `ndim` | Cantidad de ejes | `3` |
| `shape` | Tamaño por eje | `(2, 3, 3)` |
| `size` | Total de elementos | `2 * 3 * 3 = 18` |
| `dtype` | Tipo de datos | `int64` |
| `nbytes` | Memoria ocupada | `size * 8` bytes |

In [ ]:
print("Atributos:\n")
print(f"Ndim   : {muestra.ndim}")
print(f"Shape  : {muestra.shape}")
print(f"Size   : {muestra.size}")
print(f"Dtype  : {muestra.dtype}")
print(f"Nbytes : {muestra.nbytes}")

<!-- TIP -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2f81f7; background:rgba(47, 129, 247,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💡 Tip — cómo leer shape</strong>
  El orden de <code>shape</code> es el orden en que se indexa. Con <code>(días, horas, sensores)</code>, se accede como <code>muestra[dia, hora, sensor]</code>. Equivocarse de orden es el error más común al empezar.
</div>

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — size sale de shape</strong>
  <code>size</code> siempre es el producto de <code>shape</code>. Verifíquenlo sin escribir el número a mano.
</div>

In [ ]:
print(f"Size: {muestra.size}")
print("Tamaño calculado a partir de size: ", muestra.shape[0] * muestra.shape[1] * muestra.shape[2])


---

## 3. Vectorización y creación de arreglos

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — vectorización</strong>
  La vectorización es la ausencia de cualquier ciclo o indexado explícito en el código: las operaciones se expresan sobre el arreglo completo y NumPy las resuelve «detrás de escenas» con rutinas precompiladas.
</div>

### Dos formas de expresar la misma transformación

La vectorización no cambia la fórmula: cambia quién repite el trabajo y cómo lo expresamos.

```mermaid
sequenceDiagram
    autonumber
    box rgb(240, 240, 240) Python Puro
    participant Py as Intérprete Python
    end
    box rgb(220, 240, 255) NumPy
    participant NP as NumPy (Python API)
    participant C as Motor C / SIMD
    end

    Note over Py: Loop en Bytecode (N veces)
    loop Para cada elemento
        Py->>Py: 1. Inspecciona tipo de dato<br/>2. Desempaqueta PyObject<br/>3. Opera y empaqueta resultado
    end

    Note over NP, C: Operación Vectorizada (1 llamada)
    NP->>C: Pasa puntero a bloque contiguo de memoria
    C->>C: Aplica instrucción vectorial sobre el bloque entero
    C-->>NP: Retorna arreglo de memoria
```

Pasar 24 temperaturas de Celsius a Fahrenheit con Python puro exige un ciclo.
Con NumPy se escribe igual que la fórmula matemática:

In [ ]:
# -------------------------------------------------------------
# 1. Python Puro: Requiere 3 ciclos anidados para llegar al dato
# -------------------------------------------------------------
temp_f_py = []
for dia in muestra:
    dia_f = []
    for hora in dia:
        temp_c = hora[0]  # Métrica 0 = Temperatura
        dia_f.append(temp_c * 9/5 + 32)
    temp_f_py.append(dia_f)

np.array(temp_f_py)

In [ ]:
# -------------------------------------------------------------
# Extra: Nuevo arreglo el canal de temperatura en F°
# -------------------------------------------------------------
#
muestra_f = muestra.astype(float).copy()

# calculo
temp_c = muestra_f[:, :, 0]
temp_f =  temp_c * 9/5 + 32

# reasignación
muestra_f[:, :, 0] = temp_f

muestra_f

Para **crear** arreglos existen varias familias:

In [ ]:
print("Array   :", np.array([1, 2, 3]))
print("Zeros   :", np.zeros(4))
print("Ones    :", np.ones(4))
print("Arange  :", np.arange(0, 24, 6))
print("Full    :", np.full(3, 7))

Con esas piezas podemos construir una muestra determinista de una **semana
completa**. La
temperatura sigue un ciclo diario (mínimo de madrugada, máximo por la tarde) y
la humedad se mueve al revés que la temperatura:

In [ ]:
# La muestra es fija: no depende de una semilla ni de números aleatorios.
dias, horas = 7, 24
hora = np.arange(horas)

ciclo = 18 + 8 * np.sin((hora - 9) * 2 * np.pi / 24)
variacion_temperatura = np.array([-1, 0, 1, 0, 2, -2, 1])[:, None]
temperatura = np.tile(ciclo, (dias, 1)) + variacion_temperatura

variacion_humedad = np.array([0, 4, -3, 6, -5, 2, 8])[:, None]
humedad = (
    70 - 1.5 * (temperatura - 18) + variacion_humedad
)

ciclo_viento = 12 + 12 * np.abs(
    np.sin((hora - 4) * 2 * np.pi / 24)
)
variacion_viento = np.array([0, 2, -1, 3, -2, 1, 4])[:, None]
viento = np.tile(ciclo_viento, (dias, 1)) + variacion_viento

muestra = np.stack([temperatura, humedad, viento], axis=2).round().astype(int)

print("Forma:", muestra.shape, "| Tipo:", muestra.dtype)

`muestra` es el arreglo que usaremos de aquí en adelante. Los sensores están
en el último eje, siempre en este orden:

In [ ]:
SENSORES = ["Temperatura (°C)", "Humedad (%)", "Viento (km/h)"]

for i, nombre in enumerate(SENSORES):
    print(f"Índice {i} -> {nombre}")

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210, 153, 34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ Aviso — dtype homogéneo</strong>
  Un <code>ndarray</code> es de un solo tipo de datos. <code>np.array(["hola", "chao"])</code> crea un arreglo de strings sin problema, pero si las filas tienen largos distintos el arreglo es irregular y NumPy avisa con un error, porque no puede representar la estructura como un bloque rectangular.
</div>

In [ ]:
try:
    np.array([[1, 2, 3], [4, 5]])
except ValueError as error:
    print("ValueError:", error)

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — la semana tiene el tamaño esperado</strong>
  7 días × 24 horas × 3 sensores son 504 lecturas.
</div>

In [ ]:
assert muestra.shape == (7, 24, 3)
assert muestra.size == 7 * 24 * 3 == 504
print("Lecturas registradas:", muestra.size)

---

## 4. Indexado y slicing

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — slicing</strong>
  El slicing permite seleccionar una porción de un arreglo con la notación <code>inicio:fin:paso</code>. Al omitir los extremos se toma todo el eje, y un paso <code>-1</code> recorre el eje en orden inverso.
</div>

### Qué queda después de seleccionar

Cada selección responde una pregunta distinta sobre el mismo bloque. Un índice fijo elimina un eje; `:` lo conserva completo.

```mermaid
flowchart TD
    M["<b>muestra</b><br/>(2, 3, 3) — Tensor 3D"]

    M -->|"1. Selección de Plano"| A["<b>muestra[0]</b><br/>Shape: (3, 3)<br/>Corta el primer día (Eje 0 = 0)"]
    M -->|"2. Extracción de Feature"| B["<b>muestra[:, :, 1]</b><br/>Shape: (2, 3)<br/>Saca la columna 1 de todos los días/horas"]
    M -->|"3. Indexación Puntual"| C["<b>muestra[1, 2, 0]</b><br/>Valor: 24<br/>Día 1, Hora 2, Métrica 0"]
    M -->|"4. Reorganización"| D["<b>muestra[::-1]</b><br/>Shape: (2, 3, 3)<br/>Misma forma, invierte orden de días"]
```

In [ ]:
print("Una lectura      :", muestra[0, 12, 0])        # día 0, mediodía, temp.
print("Un día completo  :", muestra[0].shape)         # (24, 3)
print("Toda la humedad  :", muestra[:, :, 1].shape)   # (7, 24)
print("Los 3 sensores a las 12 del día 0:", muestra[0, 12])

Los dos puntos `:` significan «todo este eje». Por eso `muestra[:, :, 1]`
se lee como *todos los días, todas las horas, sensor 1*.

In [ ]:
# Rangos y saltos
print("Primeras 3 horas del día 0:\n", muestra[0, 0:3])
print("\nUna medición cada 6 horas del día 0:\n", muestra[0, ::6])

El paso negativo invierte un eje. Es la herramienta que van a necesitar cada
vez que tengan que **dar vuelta** un arreglo a lo largo de una dimensión:

In [ ]:
semana_al_reves = muestra[::-1]          # invierte el eje 0 (los días)
dia_al_reves = muestra[0, ::-1]          # invierte el eje 1 (las horas)

print("Primer día original :", muestra[0, 0])
print("Primer día invertido:", semana_al_reves[0, 0])
print("¿Es el último día? ", np.array_equal(semana_al_reves[0], muestra[-1]))

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — invertir dos veces vuelve al original</strong>
  Invertir un eje es su propia operación inversa. Prueben también con <code>muestra[:, ::-1]</code>, que invierte las horas en vez de los días.
</div>

In [ ]:
assert np.array_equal(muestra[::-1][::-1], muestra)
print("Invertir dos veces devuelve el arreglo original")

---

## 5. Operaciones elementwise, lógicas y saturación

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — operaciones elementwise</strong>
  Las operaciones aritméticas <code>+</code>, <code>-</code>, <code>*</code> aplican <strong>elemento a elemento</strong> sobre arreglos del mismo tamaño. El producto matricial no es elementwise: usa <code>@</code>.
</div>

In [ ]:
temperaturas = muestra[:, :, 0]

print("Original       :\n\n", temperaturas, "\n\n")
print("+2 Grados      :\n\n", (temperaturas + 2), "\n\n")
print("En Fahrenheit  :\n\n", (temperaturas * 9 / 5 + 32), "\n\n")

### Rangos válidos y desbordamiento

Muchas magnitudes tienen un rango físicamente válido. La **humedad relativa**
solo puede estar entre `0` y `100 %`: un valor de `104 %` no significa nada,
es un sensor mal calibrado.

Antes de arreglarlo, un peligro con el tipo de datos:

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210, 153, 34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ Aviso — overflow de enteros pequeños</strong>
  Un tipo entero angosto se desborda <strong>en silencio</strong>: al pasarse del máximo, vuelve a empezar por el mínimo. No hay error ni advertencia, solo un número equivocado.
</div>

In [ ]:
lectura = np.array([120], dtype="int8")   # int8 llega hasta 127
print("120 + 10 En int8 :", lectura + 10)          # ¡negativo!

lectura_ancha = np.array([120], dtype="int64")
print("120 + 10 En int64:", lectura_ancha + 10)    # correcto

La lección: **trabajen con un tipo suficientemente ancho** y corrijan el rango
ustedes, de forma explícita y controlada.

### Indexado condicional y queries

### Saturar es proyectar al intervalo válido

Los valores fuera de rango no se eliminan: se llevan al límite más cercano. Por eso la operación pierde información.

```mermaid
stateDiagram-v2
    [*] --> Entrada
    Entrada --> Bajo: valor menor que 0
    Entrada --> Valido: valor entre 0 y 100
    Entrada --> Alto: valor mayor que 100
    Bajo: fuera de rango
    Valido: se conserva
    Alto: fuera de rango
    Bajo --> SalidaBaja: asignar 0
    Valido --> SalidaValida: conservar valor
    Alto --> SalidaAlta: asignar 100
    SalidaBaja: resultado = 0
    SalidaValida: resultado = valor original
    SalidaAlta: resultado = 100
```

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — query</strong>
  Una <em>query</em> es una selección por medio de una condición lógica. Comparar un arreglo devuelve otro arreglo de <code>True</code> y <code>False</code>, que sirve tanto para <strong>leer</strong> los elementos que cumplen la condición como para <strong>escribir</strong> sobre ellos.
</div>

In [ ]:
a = np.array([1, 2, 3, 4, 5])

print("A > 3       :", a > 3)
print("A[a > 3]    :", a[a > 3])

In [ ]:
# Sobre la muestra: ¿cuántas lecturas de viento superan los 20 km/h?
viento_medido = muestra[:, :, 2]

print("Lecturas con ráfagas > 20 km/h:", (viento_medido > 20).sum())
print("Temperatura media en esas horas:",
      temperaturas[viento_medido > 20].mean().round(1))

Ahora sí, el arreglo del rango. Supongamos que el sensor de humedad
sub-reporta y hay que **calibrarlo** multiplicándolo por `1.15`. Eso empuja
algunas lecturas por sobre el `100 %`:

In [ ]:
humedad_calibrada = (muestra[:, :, 1] * 1.15).round().astype(int)

print("Máximo tras calibrar:", humedad_calibrada.max(), "%")
print("Lecturas imposibles :", (humedad_calibrada > 100).sum())

Para devolverlas al rango válido se usa **indexado condicional**: se
seleccionan las que se pasaron y se les asigna el tope.

In [ ]:
humedad_calibrada[humedad_calibrada > 100] = 100
humedad_calibrada[humedad_calibrada < 0] = 0

print("Máximo tras corregir:", humedad_calibrada.max(), "%")
print("Mínimo tras corregir:", humedad_calibrada.min(), "%")

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Idea clave — saturar con indexado condicional</strong>
  Toda operación que pueda dejar valores fuera del rango válido termina con dos líneas de la forma:<br><br><code>resultado[resultado > MAXIMO] = MAXIMO</code><br><code>resultado[resultado < MINIMO] = MINIMO</code><br><br>A esto se le llama <strong>saturar</strong>. Existe <code>np.clip</code> y hace lo mismo, pero en este laboratorio se pide el indexado condicional, porque ese patrón es uno de los objetivos.
</div>

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — después de saturar no queda nada fuera de rango</strong>
  Y algo más importante: <strong>saturar destruye información</strong>. Todas las lecturas que estaban sobre 100 quedaron pegadas al mismo valor, así que ya no se puede recuperar cuál era cuál.
</div>

In [ ]:
assert humedad_calibrada.max() <= 100
assert humedad_calibrada.min() >= 0

pegadas = (humedad_calibrada == 100).sum()
print(f"{pegadas} Lecturas quedaron pegadas en 100: perdieron su valor original")

---

## 6. Funciones universales (ufuncs)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — función universal (ufunc)</strong>
  Una ufunc es una operación aplicada elemento a elemento sobre un arreglo completo, como <code>np.exp</code>, <code>np.sqrt</code>, <code>np.sin</code>, <code>np.abs</code> o <code>np.log</code>. Reciben un arreglo y devuelven otro de la misma forma.
</div>

In [ ]:
desviacion = temperaturas - temperaturas.mean()

print("Desviaciones        :", desviacion[0, :4].round(1))
print("Resultado de np.abs (magnitud):", np.abs(desviacion)[0, :4].round(1))
print("Resultado de np.sqrt (de |x|):", np.sqrt(np.abs(desviacion))[0, :4].round(2))

De hecho ya usaron una: `np.sin` generó el ciclo diario de temperatura al
principio del notebook. Veámoslo dibujado:

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(hora, temperaturas.mean(axis=0), marker="o")
plt.title("Temperatura media por hora del día")
plt.xlabel("hora")
plt.ylabel("°C")
plt.grid(alpha=0.3)
plt.show()

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — el ciclo diario tiene sentido físico</strong>
  La hora más fría debe caer de madrugada y la más calurosa por la tarde. <code>argmin</code> y <code>argmax</code> devuelven la <em>posición</em> del mínimo y del máximo, no su valor.
</div>

In [ ]:
media_por_hora = temperaturas.mean(axis=0)

print("Hora más fría    :", media_por_hora.argmin())
print("Hora más calurosa:", media_por_hora.argmax())

assert media_por_hora.argmin() < 8
assert 12 < media_por_hora.argmax() < 20

---

## 7. Broadcasting

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — broadcasting</strong>
  El broadcasting es la forma en que NumPy opera entre arreglos de tamaños distintos: repite el arreglo más chico sobre los ejes que calzan, sin copiarlo en memoria. Es lo que permite escribir <code>arreglo * 2</code> sin escribir un ciclo.
</div>

El caso útil acá: cada sensor necesita **su propio factor de calibración**.
En vez de corregir los tres por separado, se multiplica por un vector de
largo 3, que se transmite sobre el último eje.

### El vector se alinea con el último eje

Las tres columnas representan el eje de sensores. El mismo vector se aplica a cada una de las `7 × 24` posiciones de ese eje.

```mermaid
block-beta
  columns 3
  t["temperatura<br/>× 1.02"] h["humedad<br/>× 1.15"] v["viento<br/>× 0.95"]
  t0["14 × 1.02"] h0["82 × 1.15"] v0["5 × 0.95"]
  t1["19 × 1.02"] h1["68 × 1.15"] v1["11 × 0.95"]
```

In [ ]:
# un factor por sensor: temperatura, humedad, viento
calibracion = np.array([1.02, 1.15, 0.95])

calibrada = muestra * calibracion

print("Forma de la muestra :", muestra.shape)
print("Forma del vector     :", calibracion.shape)
print("Forma del resultado  :", calibrada.shape)

**Para que el broadcasting funcione**, las formas se comparan **de derecha a
izquierda**: cada par de dimensiones debe ser igual, o una de las dos debe
valer 1.

```text
muestra     (7, 24, 3)
calibracion         (3,)   ← calza con el último eje
resultado   (7, 24, 3)
```

Si el vector no calza con el último eje, NumPy se niega:

In [ ]:
try:
    muestra * np.array([1.0, 1.0, 1.0, 1.0])   # 4 factores para 3 sensores
except ValueError as error:
    print("ValueError:", error)

<!-- TIP -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2f81f7; background:rgba(47, 129, 247,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💡 Tip — operar por canal</strong>
  Cuando necesiten aplicar un cambio distinto a cada elemento del último eje, multipliquen por un vector de ese largo en lugar de escribir tres operaciones separadas. Es más corto, más rápido y más difícil de equivocar.
</div>

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — el factor llegó a cada sensor</strong>
  La media de cada sensor debe quedar multiplicada por su propio factor.
</div>

In [ ]:
medias_antes = muestra.mean(axis=(0, 1))
medias_despues = calibrada.mean(axis=(0, 1))

print("Antes  :", medias_antes.round(2))
print("Después:", medias_despues.round(2))

np.testing.assert_allclose(medias_despues, medias_antes * calibracion)
print("Cada sensor recibió exactamente su factor")

---

## 8. Vistas y copias

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — referencia, vista y copia</strong>
  En Python los nombres son <strong>referencias</strong>: <code>b = a</code> hace que ambos apunten al mismo objeto. En NumPy, además, el slicing devuelve una <strong>vista</strong>: un arreglo nuevo que comparte la memoria del original. Solo <code>.copy()</code> crea datos independientes.
</div>

### Dos nombres, una memoria; dos copias, dos memorias

La diferencia no está en el nombre, sino en el buffer que hay detrás.

```mermaid
flowchart LR
    A["a = [1, 2, 3]"] --> M["memoria A<br/>[1, 2, 3]"]
    B["b = a"] --> M
    M --> X["b[0] = 999"]
    X --> Y["a ahora vale [999, 2, 3]"]
    C["c = a.copy()"] --> N["memoria C independiente"]
    N --> Z["c[0] cambia;<br/>a no cambia"]
```

In [ ]:
a = np.array([1, 2, 3])
b = a           # referencia: el mismo objeto
b[0] = 999

print("A después de modificar b:", a)     # ¡a también cambió!

Con slicing pasa lo mismo, y es más difícil de notar porque uno cree estar
trabajando con «otro» arreglo:

In [ ]:
lunes = muestra[0]        # esto es una VISTA, no una copia
valor_original = muestra[0, 0, 0]

lunes[0, 0] = -999

print("Valor original          :", valor_original)
print("Muestra[0, 0, 0] ahora :", muestra[0, 0, 0])
print("¿Comparten memoria?     :", np.shares_memory(lunes, muestra))

In [ ]:
# Restauramos el dato y repetimos la prueba con una copia
muestra[0, 0, 0] = valor_original

lunes_copia = muestra[0].copy()
lunes_copia[0, 0] = -999

print("Muestra[0, 0, 0]  :", muestra[0, 0, 0], "(intacto)")
print("¿Comparten memoria?:", np.shares_memory(lunes_copia, muestra))

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210, 153, 34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ Aviso — mutar la original produce bugs</strong>
  Si una función modifica el arreglo que recibe, quien la llamó se queda con los datos alterados sin darse cuenta, y las operaciones encadenadas acumulan errores muy difíciles de rastrear. Es de los bugs más caros de encontrar, porque el síntoma aparece lejos de la causa.
</div>

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Idea clave — no mutar la entrada</strong>
  Una función que transforma un arreglo debe <strong>retornar uno nuevo</strong> con una copia del resultado, y nunca escribir sobre el que recibió. Es la regla que van a aplicar en todo el mini proyecto.
</div>

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — una función que respeta la entrada</strong>
  La función de abajo suma un valor sin tocar el arreglo original. Comparen qué pasa si borran el <code>.copy()</code>.
</div>

In [ ]:
def sumar_sin_mutar(datos, cantidad):
    resultado = datos.copy()
    resultado += cantidad
    return resultado


antes = muestra.copy()
_ = sumar_sin_mutar(muestra, 5)

assert np.array_equal(muestra, antes)
print("La muestra quedó intacta después de la llamada")

---

## 9. Cambio de forma y apilado

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — reshape y apilado</strong>
  <code>reshape</code> reorganiza los mismos elementos en otra forma compatible (el <code>size</code> no cambia). <code>np.stack</code> junta varios arreglos de igual forma a lo largo de un <strong>eje nuevo</strong>.
</div>

### Apilar agrega un eje; `reshape` reorganiza los existentes

Las tres matrices de sensores comparten `(7, 24)`. Al apilarlas aparece un tercer eje; al aplanar, el total de elementos permanece igual.

```mermaid
flowchart LR
    T["temperatura<br/>(7, 24)"] --> S["np.stack(axis=2)"]
    H["humedad<br/>(7, 24)"] --> S
    V["viento<br/>(7, 24)"] --> S
    S --> E["muestra<br/>(7, 24, 3)"]
    E --> R["reshape(168, 3)"]
    R --> Z["mismo size<br/>504 elementos"]
```

In [ ]:
plano = muestra.reshape(7 * 24, 3)     # una fila por lectura horaria

print("Forma original:", muestra.shape)
print("Forma aplanada:", plano.shape)
print("Mismo total   :", plano.size == muestra.size)

`np.stack` es la operación inversa del último eje: así fue como se armó
`muestra` a partir de las tres matrices de sensores.

In [ ]:
rearmada = np.stack([temperatura, humedad, viento], axis=2)

print("Temperatura:", temperatura.shape)
print("Humedad    :", humedad.shape)
print("Viento     :", viento.shape)
print("Apiladas   :", rearmada.shape, "<- Apareció un tercer eje de largo 3")

<!-- TIP -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2f81f7; background:rgba(47, 129, 247,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💡 Tip — el axis de np.stack</strong>
  El parámetro <code>axis</code> dice <strong>dónde</strong> se inserta el eje nuevo. Con <code>axis=2</code> queda al final: <code>(7, 24)</code> + <code>(7, 24)</code> + <code>(7, 24)</code> producen <code>(7, 24, 3)</code>. Con <code>axis=0</code> quedaría <code>(3, 7, 24)</code>.
</div>

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — apilar y volver a separar</strong>
  Apilar tres matrices y luego tomar el sensor <code>i</code> debe devolver exactamente la matriz <code>i</code> de partida.
</div>

In [ ]:
assert np.array_equal(rearmada[:, :, 0], temperatura)
assert np.array_equal(rearmada[:, :, 1], humedad)
assert np.array_equal(rearmada[:, :, 2], viento)

print("Apilar y separar son operaciones inversas")
print("Con axis=0 la forma sería:", np.stack([temperatura, humedad, viento], axis=0).shape)

---

## 10. Estadísticas y ejes

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — reducción por eje</strong>
  Funciones como <code>sum</code>, <code>min</code>, <code>max</code>, <code>mean</code> o <code>std</code> <strong>reducen</strong> un arreglo: consumen uno o más ejes y devuelven algo más chico. El parámetro <code>axis</code> indica cuáles.
</div>

Este es el concepto que más cuesta y el que más rinde. La clave: **`axis` es el
eje que desaparece**.

| Llamada | Qué desaparece | Forma resultante | Qué responde |
|---|---|---|---|
| `muestra.mean()` | todos | `()` | un número global |
| `muestra.mean(axis=0)` | los días | `(24, 3)` | perfil de un día promedio |
| `muestra.mean(axis=1)` | las horas | `(7, 3)` | resumen por día |
| `muestra.mean(axis=(0, 1))` | días y horas | `(3,)` | **un valor por sensor** |

### Una reducción hace desaparecer los ejes elegidos

Primero miremos una matriz de dos dimensiones. La misma regla se extiende a `muestra`: el eje elegido es el que se resume.

```mermaid
flowchart LR
    subgraph M["matriz (2, 3)"]
        direction TB
        R0["1 · 2 · 3"]
        R1["4 · 5 · 6"]
    end
    M --> A["axis=0<br/>desaparecen filas<br/>resultado (3,)"]
    M --> B["axis=1<br/>desaparecen columnas<br/>resultado (2,)"]
    M --> C["sin axis<br/>desaparecen todos<br/>resultado ( )"]
```

In [ ]:
print("Global        :", muestra.mean().round(2), "-> Forma", muestra.mean().shape)
print("Axis=0        -> Forma", muestra.mean(axis=0).shape)
print("Axis=1        -> Forma", muestra.mean(axis=1).shape)
print("Axis=(0, 1)   -> Forma", muestra.mean(axis=(0, 1)).shape)

La última es la más útil: **agrega sobre todo salvo el último eje**, así que
deja un valor por sensor. Es la forma de resumir una medición sin mirarla.

In [ ]:
for i, nombre in enumerate(SENSORES):
    canal = muestra[:, :, i]
    print(
        f"{nombre:18s} min={canal.min():4d}  max={canal.max():4d}  "
        f"media={canal.mean():6.1f}  desv={canal.std():5.1f}"
    )

In [ ]:
# Lo mismo, sin ciclo, gracias a axis=(0, 1)
print("Mínimos   :", muestra.min(axis=(0, 1)))
print("Máximos   :", muestra.max(axis=(0, 1)))
print("Medias    :", muestra.mean(axis=(0, 1)).round(1))
print("Percentil 25:", np.percentile(muestra, 25, axis=(0, 1)))

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — el ciclo y la reducción dan lo mismo</strong>
  El máximo por sensor calculado con un ciclo debe coincidir, valor por valor, con <code>muestra.max(axis=(0, 1))</code>. Si coinciden, entendieron <code>axis</code>.
</div>

In [ ]:
con_ciclo = np.array([muestra[:, :, i].max() for i in range(3)])
con_axis = muestra.max(axis=(0, 1))

print("Con ciclo:", con_ciclo)
print("Con axis :", con_axis)

assert np.array_equal(con_ciclo, con_axis)

---

## De la muestra a la imagen

Todo lo anterior se hizo sobre datos de sensores. Acá viene el punto: **una
fotografía a color es exactamente el mismo tipo de objeto**.

| | Muestra | Imagen a color |
|---|---|---|
| Forma | `(7, 24, 3)` | `(alto, ancho, 3)` |
| Eje 0 | días | filas de píxeles |
| Eje 1 | horas | columnas de píxeles |
| Eje 2 | 3 sensores | 3 canales RGB |
| Un elemento | una lectura | la intensidad de un color |
| Rango válido | humedad en `[0, 100]` | intensidad en `[0, 255]` |
| `axis=(0, 1)` | un valor por sensor | un valor por canal |
| Vector de largo 3 | calibrar cada sensor | ajustar cada color |

Cambia el significado de los ejes. **No cambia ninguna de las técnicas.**

### El mismo tensor con otro significado

No es una transformación paso a paso: es una correspondencia de significados.

```mermaid
mindmap
  root((Tensor de tres ejes))
    Muestra
      eje 0: días
      eje 1: horas
      eje 2: sensores
    Imagen RGB
      eje 0: filas
      eje 1: columnas
      eje 2: canales R, G y B
    Mismas operaciones
      slicing
      broadcasting
      axis=(0, 1)
```

Podemos construir una imagen chiquita a mano y mirarla, para convencernos de
que es un arreglo como cualquier otro:

In [ ]:
bandera = np.zeros((6, 9, 3), dtype=int)

bandera[0:3, :, 0] = 255      # mitad superior: canal rojo al máximo
bandera[3:6, :, 2] = 255      # mitad inferior: canal azul al máximo

print("Forma:", bandera.shape, "| Tipo:", bandera.dtype)

plt.imshow(bandera)
plt.axis("off")
plt.show()

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45, 164, 78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación — leer la imagen como tensor</strong>
  Sin volver a mirar el dibujo: el canal verde quedó entero en 0, y el píxel de la esquina superior izquierda es rojo puro.
</div>

In [ ]:
assert bandera[:, :, 1].max() == 0
assert np.array_equal(bandera[0, 0], [255, 0, 0])

print("Un valor por canal:", bandera.max(axis=(0, 1)))

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — cómo se representa una imagen fuera de este laboratorio</strong>
  En el ecosistema científico conviven <strong>dos</strong> representaciones estándar. La de <strong>almacenamiento y lectura</strong> es <code>uint8</code> en el rango <code>[0, 255]</code>: un byte por canal, y es lo que devuelven PIL, OpenCV e imageio al abrir un archivo. La de <strong>cómputo</strong> es <code>float32</code> o <code>float64</code> en el rango <code>[0.0, 1.0]</code>: la usan scikit-image, matplotlib cuando recibe flotantes, y las librerías de aprendizaje profundo. El flujo habitual alterna entre las dos: se carga en <code>uint8</code>, se convierte a flotante para operar y se vuelve a <code>uint8</code> para mostrar o guardar.<br><br>En el mini proyecto van a trabajar con <code>int</code> en <code>[0, 255]</code> por una razón didáctica: con <code>uint8</code> la suma desborda en silencio —como vieron con <code>int8</code> en la sección 5— y no quedaría nada que saturar; con flotantes en <code>[0, 1]</code> la saturación se vuelve casi invisible. El <code>int</code> es el punto donde el desbordamiento no ocurre y corregir el rango tiene que ser una decisión <strong>explícita</strong> de ustedes.<br><br>Lo que sí es estándar en esa representación es el resto: la forma <code>(alto, ancho, canales)</code> —conocida como <em>channels-last</em>— y el orden RGB. Vale la pena saber que PyTorch invierte la forma a <code>(canales, alto, ancho)</code> y que OpenCV lee los canales en orden <strong>BGR</strong>: son las dos confusiones más frecuentes al pasar de una librería a otra.
</div>

<!-- TIP -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2f81f7; background:rgba(47, 129, 247,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">💡 Tip — floats en imshow</strong>
  <code>matplotlib</code> interpreta un arreglo flotante en el rango <code>[0.0, 1.0]</code>. Si le pasan valores de 0 a 255 como <code>float</code>, la imagen aparece completamente blanca. Manténganlas como <code>int</code>.
</div>

## Del tensor a PixelLab

Este es el recorrido que van a observar después en la app. Streamlit coordina la interacción; las operaciones de NumPy viven dentro de PixelLab.

```mermaid
sequenceDiagram
    actor U as Usuario
    participant S as Streamlit
    participant P as PixelLab
    participant N as NumPy
    U->>S: selecciona imagen y filtros
    S->>S: vuelve a ejecutar la app
    S->>P: entrega Imagen y pipeline
    P->>N: aplica slicing, broadcasting y operaciones
    N-->>P: devuelve un arreglo nuevo
    P-->>S: entrega Imagen resultado
    S-->>U: muestra original, resultado e histogramas
```

---

## Anexo: referencia rápida

Todo lo que se usó en este notebook, en un solo lugar.

In [ ]:
# Creación
np.array([1, 2, 3]); np.zeros((4, 3)); np.ones((4, 3)); np.arange(0, 10, 2)

# Propiedades
muestra.ndim; muestra.shape; muestra.size; muestra.dtype

# Indexado y slicing
muestra[0, 12, 0]      # un elemento
muestra[:, :, 1]       # todo un canal del último eje
muestra[0, 0:3]        # un rango
muestra[::-1]          # invertir un eje

# Indexado condicional (saturación)
copia = muestra.copy()
copia[copia > 100] = 100
copia[copia < 0] = 0

# Ufuncs
np.abs(copia); np.sqrt(np.abs(copia)); np.sin(copia)

# Broadcasting
muestra * 2
muestra * np.array([1.02, 1.15, 0.95])     # un factor por elemento del eje 2

# Vista vs copia
vista = muestra[0]             # comparte memoria
independiente = muestra[0].copy()

# Cambio de forma y apilado
muestra.reshape(7 * 24, 3)
np.stack([temperatura, humedad, viento], axis=2)

# Reducciones por eje
muestra.max(axis=(0, 1))
np.percentile(muestra, 25, axis=(0, 1))

print("Referencia ejecutada sin errores")

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240, 136, 62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Siguiente paso — el mini proyecto</strong>
  Con esto ya tienen todas las técnicas que necesita el laboratorio. Abran <code>Lab3_Enunciado.ipynb</code>: ahí construyen <strong>PixelLab</strong>, una librería de procesamiento de imágenes, y la prueban en vivo desde una app. Ninguna operación del proyecto está resuelta en este notebook — lo que aprendieron acá es cómo hacerlas.
</div>